# Predicción de spinodoides con modelos entrenados

Este notebook carga los pesos almacenados en `modelos/` y no reentrena nada. Permite usar ambos sentidos del modelo:

- **Diseño → geometría**: `Beta`, `Rho` y tres ángulos de anisotropía producen los siete descriptores geométricos.
- **Geometría → diseño**: los siete descriptores geométricos producen un candidato puntual y tres candidatos de la MDN.

Ejecuta las celdas de arriba hacia abajo. Para nuevas consultas solo cambia los valores de una de las dos últimas celdas y vuelve a ejecutarla.

> Nota: el diseño inverso no es único. En especial, los ángulos pueden variar mucho sin cambiar de manera importante estos siete descriptores escalares. Por eso la MDN devuelve varios candidatos, ordenados por su probabilidad estimada.

## Dependencias

Desde la raíz del repositorio, instala una vez las dependencias con `pip install -r requirements.txt`. El notebook debe abrirse desde esa misma raíz, donde existen las carpetas `modelos/` y el archivo `requirements.txt`.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

ROOT = Path.cwd()
MODELOS = ROOT / 'modelos'
if not MODELOS.is_dir():
    raise FileNotFoundError(
        'No encuentro la carpeta modelos/. Abre este notebook desde la raíz del repositorio p_final.'
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Carpeta del proyecto: {ROOT.resolve()}')
print(f'Dispositivo: {device}')

In [ ]:
# Arquitecturas: deben coincidir exactamente con los pesos guardados.
class ForwardNN(nn.Module):
    def __init__(self, input_dim=5, output_dim=7, hidden=(512, 256, 256, 128), dropout=0.1):
        super().__init__()
        layers, previous = [], input_dim
        for width in hidden:
            layers.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)])
            previous = width
        layers.append(nn.Linear(previous, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ForwardNNProb(nn.Module):
    def __init__(self, input_dim=5, output_dim=7, hidden=(512, 256, 256, 128), dropout=0.1):
        super().__init__()
        layers, previous = [], input_dim
        for width in hidden:
            layers.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)])
            previous = width
        self.backbone = nn.Sequential(*layers)
        self.mu_head = nn.Linear(previous, output_dim)
        self.logvar_head = nn.Linear(previous, output_dim)

    def forward(self, x):
        h = self.backbone(x)
        return self.mu_head(h), self.logvar_head(h)


class InverseNN(nn.Module):
    def __init__(self, input_dim=7, output_dim=5, hidden=(128, 128, 64), dropout=0.1):
        super().__init__()
        layers, previous = [], input_dim
        for width in hidden:
            layers.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)])
            previous = width
        layers.append(nn.Linear(previous, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class MDN(nn.Module):
    """Mezcla de gaussianas: geometría (7) → parámetros de diseño (5)."""
    def __init__(self, input_dim=7, output_dim=5, K=3, hidden=(128, 128, 64), dropout=0.1):
        super().__init__()
        self.K, self.D = K, output_dim
        layers, previous = [], input_dim
        for width in hidden:
            layers.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)])
            previous = width
        self.body = nn.Sequential(*layers)
        self.pi_head = nn.Linear(previous, K)
        self.mu_head = nn.Linear(previous, K * output_dim)
        self.logvar_head = nn.Linear(previous, K * output_dim)

    def forward(self, x):
        h = self.body(x)
        logits = self.pi_head(h)
        mu = self.mu_head(h).view(-1, self.K, self.D)
        log_var = self.logvar_head(h).view(-1, self.K, self.D)
        return logits, mu, torch.clamp(log_var, -7.0, 2.0)

In [ ]:
# Carga de normalizadores, configuración y pesos. No se ejecuta entrenamiento.
with open(MODELOS / 'config.json', encoding='utf-8') as file:
    config = json.load(file)

scalers = joblib.load(MODELOS / 'scalers.pkl')
scaler_X = scalers['scaler_X']
scaler_y = scalers['scaler_y']
theta_cols = list(scalers['theta_cols'])
G_cols = list(scalers['G_cols'])

def load_state(path):
    # weights_only está disponible en PyTorch reciente; el fallback mantiene compatibilidad.
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)

f_nn = ForwardNN(**config['f_nn']).to(device)
f_nn.load_state_dict(load_state(MODELOS / 'f_nn.pt'))

f_nn_prob = ForwardNNProb(**config['f_nn_prob']).to(device)
f_nn_prob.load_state_dict(load_state(MODELOS / 'f_nn_prob.pt'))

i_nn = InverseNN(**config['i_nn']).to(device)
i_nn.load_state_dict(load_state(MODELOS / 'i_nn.pt'))

mdn = MDN(**config['mdn_k3']).to(device)
mdn.load_state_dict(load_state(MODELOS / 'mdn_k3.pt'))

for model in (f_nn, f_nn_prob, i_nn, mdn):
    model.eval()

print('Modelos cargados correctamente.')
print('Parámetros de diseño:', theta_cols)
print('Descriptores geométricos:', G_cols)

In [ ]:
# Funciones de consulta. Los valores de entrada y salida siempre están en escala física.
THETA_LIMITS = {
    'Beta': (5 * np.pi, 15 * np.pi),
    'Rho': (0.3, 0.7),
    'Aniso_X': (0.0, 89.9),
    'Aniso_Y': (0.0, 89.9),
    'Aniso_Z': (0.0, 89.9),
}

def _as_row(values, expected_size, label):
    row = np.asarray(values, dtype=float).reshape(1, -1)
    if row.shape[1] != expected_size:
        raise ValueError(f'{label} debe contener exactamente {expected_size} valores; se recibieron {row.shape[1]}.')
    if not np.isfinite(row).all():
        raise ValueError(f'{label} contiene valores no numéricos o no finitos.')
    return row

def advertencias_diseno(theta):
    warnings = []
    for value, name in zip(theta, theta_cols):
        low, high = THETA_LIMITS[name]
        if not low <= value <= high:
            warnings.append(f'{name}={value:.4g} está fuera del rango de entrenamiento [{low:.4g}, {high:.4g}].')
    return warnings

def predecir_geometria(theta):
    """Theta = [Beta, Rho, Aniso_X, Aniso_Y, Aniso_Z] → tabla de G."""
    theta = _as_row(theta, len(theta_cols), 'Theta')
    theta_n = scaler_X.transform(theta)
    theta_t = torch.tensor(theta_n, dtype=torch.float32, device=device)
    with torch.no_grad():
        g_fnn_n = f_nn(theta_t).cpu().numpy()
        mu_n, logvar_n = f_nn_prob(theta_t)
        mu_n = mu_n.cpu().numpy()
        sigma_n = np.exp(0.5 * logvar_n.cpu().numpy())

    return pd.DataFrame({
        'Propiedad': G_cols,
        'f-NN': scaler_y.inverse_transform(g_fnn_n)[0],
        'f-NN probabilística (media)': scaler_y.inverse_transform(mu_n)[0],
        'Desv. est. del modelo': sigma_n[0] * scaler_y.scale_,
    })

def predecir_diseno(geometria):
    """G = 7 descriptores geométricos → candidato puntual y modos MDN."""
    g = _as_row(geometria, len(G_cols), 'La geometría objetivo')
    g_n = scaler_y.transform(g)
    g_t = torch.tensor(g_n, dtype=torch.float32, device=device)

    with torch.no_grad():
        theta_i_nn = i_nn(g_t).cpu().numpy()
        logits, mu_n, _ = mdn(g_t)
        weights = torch.softmax(logits, dim=1).cpu().numpy()[0]
        theta_modes = mu_n.cpu().numpy()[0]

        candidates_n = np.vstack([theta_i_nn, theta_modes])
        g_reconstructed_n = f_nn(torch.tensor(candidates_n, dtype=torch.float32, device=device)).cpu().numpy()

    candidates = scaler_X.inverse_transform(candidates_n)
    reconstruction_mse = ((g_reconstructed_n - g_n) ** 2).mean(axis=1)
    order = np.argsort(-weights)
    rows = [
        ['i-NN (puntual)', np.nan, *candidates[0], reconstruction_mse[0]]
    ]
    for rank, index in enumerate(order, start=1):
        rows.append([f'MDN modo {rank}', weights[index], *candidates[1 + index], reconstruction_mse[1 + index]])

    return pd.DataFrame(rows, columns=[
        'Candidato', 'Peso MDN', *theta_cols, 'MSE de reconstrucción (normalizada)'
    ])

def verificar_candidato(theta):
    """Devuelve los siete descriptores estimados para un candidato de diseño."""
    return predecir_geometria(theta)[['Propiedad', 'f-NN']].rename(columns={'f-NN': 'Geometría reconstruida'})

## 1. Diseño → geometría

Edita los cinco números de `theta`. Los rangos usados para el entrenamiento fueron `Beta ∈ [5π, 15π]`, `Rho ∈ [0.3, 0.7]` y cada anisotropía entre `0°` y `89.9°`.

In [ ]:
# [Beta, Rho, Aniso_X, Aniso_Y, Aniso_Z]
theta = [30.0, 0.50, 45.0, 45.0, 45.0]

warnings = advertencias_diseno(theta)
if warnings:
    print('Advertencia de extrapolación:')
    for warning in warnings:
        print(' -', warning)

display(predecir_geometria(theta).round(4))

## 2. Geometría → diseño

Edita los siete números de `geometria_objetivo` respetando este orden: `Area_Inner`, `Area_Outer`, `Total_Area`, `Volume`, `SA_Vol_Ratio`, `Porosity`, `Pore_Size`.

Los modos MDN son alternativas válidas propuestas por el modelo. Para cada una se informa el error al pasarla nuevamente por la red directa: un valor menor indica mejor reproducción de la geometría objetivo en el espacio normalizado.

In [ ]:
# [Area_Inner, Area_Outer, Total_Area, Volume, SA_Vol_Ratio, Porosity, Pore_Size]
# Sustituye este ejemplo por tus siete descriptores geométricos.
geometria_objetivo = [1.0, 1.0, 2.0, 0.4, 5.0, 0.5, 0.12]

candidatos = predecir_diseno(geometria_objetivo)
display(candidatos.round(4))

# Para comprobar manualmente cualquier candidato, copia sus cinco valores:
# verificar_candidato(candidatos.loc[0, theta_cols].to_numpy())